# Initialization

In [0]:
from bronze_config import INGESTION_CONFIG
import uuid
from datetime import datetime

In [0]:
# Job Logging Info (starts here...)

start_time = datetime.now()

In [0]:
%sql
USE CATALOG salesdatalakehouse;
USE SCHEMA bronze;

### Read from CSV and Write Delta Table

In [0]:
for file in INGESTION_CONFIG:
    # Load data from Volume using PySpark
    df = spark.read.csv(
        file["path"],
        header=True,
        inferSchema=True
    )

    # Write data as a delta table
    df.write.format("delta").mode("overwrite").saveAsTable(f'{file["table"]}')

## Raw Data Ingestion Logging

In [0]:

# Job Logging Info (ends here...)
run_id = str(uuid.uuid4())
job_name = "sales_etl"
task_name = 'load_Bronze'
schema_name = 'bronze'
table_name = 'NULL'

end_time = datetime.now()

# Logging into pipeline_runs table
spark.sql(f"""
  INSERT INTO salesdatalakehouse.audit.pipeline_runs
  VALUES (
    '{run_id}',
    '{job_name}',
    '{task_name}',
    '{schema_name}',
    NULL,
    '{start_time}',
    '{end_time}',
    DATEDIFF(SECOND, '{start_time}', '{end_time}'),
    'SUCCESS',
    0,
    0,
    0,
    NULL,
    CURRENT_TIMESTAMP()
  )
""")

print(f"✓ Logged run {run_id} to audit table")

In [0]:
# Get all file names from Volume folders

# BASE_PATH = '/Volumes/salesdatalakehouse/bronze/sourcesystems'
# folders = dbutils.fs.ls(f'{BASE_PATH}')
# for folder in folders:
#     files = dbutils.fs.ls(f'{BASE_PATH}/{folder.name}/')
    # for file in files:
    #     print(file.name)
